Evaluating models with the methods in the research paper but with a subject independant classification

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from scipy import signal
from tqdm import tqdm

# Machine Learning Classifiers from the paper
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import RadiusNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Global Configurations
WINDOW_SIZE = 250
STEP_SIZE = 50
LABELED_DIR = Path("./filtered/csv_labeled").resolve()

In [2]:
def extract_paper_features(x, fs=1000):
    """
    Extracts the 7 time and frequency domain features defined in the study.
    x: 1D numpy array of 250 samples (one EMG channel)
    """
    # 1. Max Value (Vmax): Maximum absolute amplitude
    vmax = np.max(np.abs(x))
    
    # 2. Integrated EMG (IEMG): Sum of absolute values
    iemg = np.sum(np.abs(x))
    
    # 3. Mean Absolute Value (MAV): Average of absolute values
    mav = np.mean(np.abs(x))
    
    # 4. Simple Square Integral (SSI): Energy of the signal
    ssi = np.sum(x**2)
    
    # 5. Variance (VAR): Power of the signal
    var = np.var(x, ddof=1) if len(x) > 1 else 0
    
    # 6. Root Mean Square (RMS)
    rms = np.sqrt(np.mean(x**2))
    
    # 7. Mean Frequency (MNF)
    # Calculated using Welch's method for power spectral density
    f, Pxx = signal.welch(x, fs=fs, nperseg=len(x))
    mnf = np.sum(f * Pxx) / np.sum(Pxx) if np.sum(Pxx) != 0 else 0
    
    return [vmax, iemg, mav, ssi, var, rms, mnf]

In [3]:
def load_and_extract_features(labeled_dir: Path, window_size=250, step_size=50):
    all_files = sorted(labeled_dir.glob("*.csv"))
    if not all_files:
        raise FileNotFoundError(f"No CSV files found in {labeled_dir}")
        
    train_files = all_files[:32]   # Subjects 1-32
    val_files   = all_files[32:36] # Subjects 33-36
    test_files  = all_files[36:40] # Subjects 37-40
    
    def process_files(file_list):
        X_list, y_list = [], []
        
        for file_path in file_list:
            print(f"   -> Extracting Features: {file_path.name} ...")
            df = pd.read_csv(file_path, header=None)
            
            raw_emg = df.iloc[:, 0:4].values.astype(np.float32)
            labels_column = df.iloc[:, 4].values
            data = np.hstack((raw_emg, labels_column.reshape(-1, 1)))
            
            num_windows = (len(data) - window_size) // step_size + 1
            
            for i in range(num_windows):
                start = i * step_size
                end   = start + window_size
                
                # Fast O(1) Boundary Check
                if data[start, 4] != data[end - 1, 4]:
                    continue
                
                window_data = data[start:end, 0:4]
                window_features = []
                
                # Extract the 7 features for all 4 channels
                for ch in range(4):
                    ch_features = extract_paper_features(window_data[:, ch], fs=1000)
                    window_features.extend(ch_features)
                
                # Append flat 28-feature vector
                X_list.append(window_features)
                y_list.append(int(data[start, 4]))
                
        return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64)

    print("--- Processing TRAINING Data ---")
    X_train, y_train = process_files(train_files)
    
    print("\n--- Processing VALIDATION Data ---")
    X_val, y_val = process_files(val_files)
    
    print("\n--- Processing TESTING Data ---")
    X_test, y_test = process_files(test_files)
    
    # Zero-index labels if necessary
    if y_train.min() == 1:
        y_train -= 1
        y_val   -= 1
        y_test  -= 1
        
    return X_train, y_train, X_val, y_val, X_test, y_test

# Execute feature extraction
X_train, y_train, X_val, y_val, X_test, y_test = load_and_extract_features(LABELED_DIR, WINDOW_SIZE, STEP_SIZE)

# Standardize features (Critical for SVM and distance-based classifiers)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\nFinal Training Matrix Shape: {X_train_scaled.shape} (Windows x Features)")

--- Processing TRAINING Data ---
   -> Extracting Features: 10_filtered.csv ...
   -> Extracting Features: 11_filtered.csv ...
   -> Extracting Features: 12_filtered.csv ...
   -> Extracting Features: 13_filtered.csv ...
   -> Extracting Features: 14_filtered.csv ...
   -> Extracting Features: 15_filtered.csv ...
   -> Extracting Features: 16_filtered.csv ...
   -> Extracting Features: 17_filtered.csv ...
   -> Extracting Features: 18_filtered.csv ...
   -> Extracting Features: 19_filtered.csv ...
   -> Extracting Features: 1_filtered.csv ...
   -> Extracting Features: 20_filtered.csv ...
   -> Extracting Features: 21_filtered.csv ...
   -> Extracting Features: 22_filtered.csv ...
   -> Extracting Features: 23_filtered.csv ...
   -> Extracting Features: 24_filtered.csv ...
   -> Extracting Features: 25_filtered.csv ...
   -> Extracting Features: 26_filtered.csv ...
   -> Extracting Features: 27_filtered.csv ...
   -> Extracting Features: 28_filtered.csv ...
   -> Extracting Features: 2

In [ ]:
# Initialize the five classifiers from the paper
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=50, random_state=42),
    "Support Vector Machine": SVC(kernel='rbf', random_state=42),
    "Radius Neighbors": RadiusNeighborsClassifier(radius=15.0, outlier_label=0, n_jobs=-1) 
}

# Dictionary to store results
results = {}

print("="*60)
print("TRAINING AND EVALUATING PAPER MODELS (Subject-Independent)")
print("="*60)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Predict on the unseen Subject 37-40 test set
    y_pred = model.predict(X_test_scaled)
    
    # Calculate Accuracy
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    
    print(f"--> {name} Test Accuracy: {acc * 100:.2f}%")

print("\n" + "="*60)
print("FINAL SUBJECT-INDEPENDENT COMPARISON")
print("="*60)
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:<25}: {acc * 100:.2f}%")

TRAINING AND EVALUATING PAPER MODELS (Subject-Independent)

Training Decision Tree...
--> Decision Tree Test Accuracy: 20.05%

Training Random Forest...
--> Random Forest Test Accuracy: 24.55%

Training Gradient Boosting...
